# CheMLFlow Tutorial 00: Dataset EDA

This notebook shows the simplest useful CheMLFlow workflow:

- load a local CSV
- run the generic EDA node
- inspect the generated plots

For this example, we use the bundled tutorial PGP dataset.

## 1. Choose the repo to clone

The default points at your `origin` fork and the current tutorial branch. Override `CHEMLFLOW_TUTORIAL_REPO` or `CHEMLFLOW_TUTORIAL_BRANCH` if you want a different source.

In [ ]:
import os

REPO_URL = os.environ.get("CHEMLFLOW_TUTORIAL_REPO", "https://github.com/bsmith24/CheMLFlow.git")
REPO_BRANCH = os.environ.get("CHEMLFLOW_TUTORIAL_BRANCH", "phase2_doe_support")
print(REPO_URL)
print(REPO_BRANCH)

## 2. Clone the repo and move into it

In [ ]:
%cd /content
!rm -rf CheMLFlow
!git clone --branch "$REPO_BRANCH" --single-branch "$REPO_URL"
%cd /content/CheMLFlow

## 3. Install the minimum dependencies for this tutorial

This tutorial only needs the packages required to import `main.py`, load the CSV, generate the EDA plots, and build a few RDKit-based chemistry views.

In [ ]:
!python -m pip install --upgrade pip
!python -m pip install "PyYAML>=6.0.1,<7" "pandas>=2.2.3,<2.3" "numpy>=1.26,<2.1" "scipy>=1.15.2,<1.16" "scikit-learn>=1.6.1,<1.7" "matplotlib>=3.10.1,<3.11" "seaborn>=0.13.2,<0.14" "joblib>=1.4,<2" "xgboost>=3.0.0,<3.1" "pydantic>=2.9,<3" "rdkit>=2024.9"

## 4. Write the config

Build the config as a normal Python object, then save it as YAML.

In [ ]:
from pathlib import Path
import yaml

CONFIG_PATH = Path("tutorials/00_dataset_eda/configs/pgp_raw_eda_colab.yaml")
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)

config = {
    "global": {
        "pipeline_type": "pgp",
        "task_type": "classification",
        "base_dir": "tutorials/00_dataset_eda/artifacts/data",
        "run_dir": "tutorials/00_dataset_eda/artifacts/run",
        "target_column": "Activity",
        "random_state": 42,
        "thresholds": {"active": 1000, "inactive": 10000},
    },
    "pipeline": {
        "nodes": ["get_data", "analyze.eda"],
    },
    "get_data": {
        "data_source": "local_csv",
        "source": {"path": "tutorials/data/pgp_broccatelli.csv"},
    },
    "analyze": {
        "eda": {
            "include": {
                "overview": True,
                "missingness": True,
                "numeric_histograms": True,
                "correlation_heatmap": True,
                "target_distribution": True,
                "class_balance": True,
                "descriptor_scatter": False,
                "descriptor_boxplots": False,
            }
        }
    },
}

config_yaml = yaml.safe_dump(config, sort_keys=False)
CONFIG_PATH.write_text(config_yaml, encoding="utf-8")
print(config_yaml)

## 5. Launch CheMLFlow

In [ ]:
import os
import subprocess
import sys

os.environ["CHEMLFLOW_CONFIG"] = str(CONFIG_PATH)
os.environ["MPLBACKEND"] = "Agg"
os.environ["MPLCONFIGDIR"] = "/content/.mplconfig"

result = subprocess.run([sys.executable, "main.py"], text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()

## 6. Inspect the manifest and generated files

In [ ]:
from pathlib import Path
import json

eda_dir = Path("tutorials/00_dataset_eda/artifacts/run/eda")
manifest = json.loads((eda_dir / "eda_manifest.json").read_text())
print(json.dumps(manifest, indent=2))
print("\nGenerated files:")
for path in sorted(eda_dir.iterdir()):
    print(path.name)

## 7. Molecular gallery

In [ ]:
import base64
import math
from io import BytesIO

import pandas as pd
from IPython.display import HTML, Image, display
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, Lipinski, QED

raw_df = pd.read_csv("tutorials/data/pgp_broccatelli.csv")
smiles_column = next((col for col in ["SMILES", "smiles", "Smiles", "canonical_smiles"] if col in raw_df.columns), None)
target_column = "Activity" if "Activity" in raw_df.columns else None
if smiles_column is None:
    raise ValueError("No SMILES column found in tutorial dataset.")

gallery_df = raw_df.copy()
gallery_df["mol"] = gallery_df[smiles_column].astype(str).apply(Chem.MolFromSmiles)
gallery_df = gallery_df[gallery_df["mol"].notna()].copy()
gallery_df["canonical_smiles"] = gallery_df["mol"].apply(Chem.MolToSmiles)
gallery_df = gallery_df.drop_duplicates(subset=["canonical_smiles"]).reset_index(drop=True)

def _mol_png_base64(mol, size=(320, 200)):
    image = Draw.MolToImage(mol, size=size)
    buffer = BytesIO()
    image.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("ascii")

def render_gallery(df, page=1, page_size=12):
    total = len(df)
    total_pages = max(1, math.ceil(total / page_size))
    page = min(max(1, int(page)), total_pages)
    start = (page - 1) * page_size
    end = start + page_size
    page_df = df.iloc[start:end]
    cards = []
    for idx, row in page_df.iterrows():
        img = _mol_png_base64(row["mol"])
        target_note = f"Target: {row[target_column]}" if target_column else ""
        smiles_note = row["canonical_smiles"][:42] + ("..." if len(row["canonical_smiles"]) > 42 else "")
        cards.append(
            f"<div style='background:#111827;border:1px solid #1f2937;border-radius:16px;padding:12px;'>"
            f"<img src='data:image/png;base64,{img}' style='width:100%;border-radius:10px;background:white;'/>"
            f"<div style='padding-top:10px;font-family:monospace;color:#e5e7eb;font-size:12px;'>#{idx + 1}<br>{smiles_note}<br>{target_note}</div>"
            f"</div>"
        )
    html = f"""
    <div style='margin:8px 0 18px 0;'>
      <div style='font-size:16px;font-weight:600;'>Molecule gallery</div>
      <div style='margin-top:4px;color:#6b7280;'>Showing page {page} of {total_pages} ({len(page_df)} molecules on this page, {total} valid unique molecules total).</div>
    </div>
    <div style='display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:16px;'>
      {''.join(cards)}
    </div>
    """
    display(HTML(html))

PAGE = 1
PAGE_SIZE = 12
render_gallery(gallery_df, page=PAGE, page_size=PAGE_SIZE)
print("Set PAGE to another value and rerun this cell to paginate the gallery.")

## 8. Raw data analysis

In [ ]:
from IPython.display import display

print(f"Rows: {len(raw_df)}")
print(f"Columns: {len(raw_df.columns)}")
print(f"SMILES column: {smiles_column}")
print(f"Target column: {target_column}")
display(raw_df.head(8))

missing_summary = raw_df.isna().sum().sort_values(ascending=False)
missing_summary = missing_summary[missing_summary > 0]
if missing_summary.empty:
    print("No missing values detected in the raw tutorial dataset.")
else:
    display(missing_summary.head(10).rename("missing_count").to_frame())

In [ ]:
for name in ["dataset_overview.png", "missingness.png"]:
    path = eda_dir / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))

## 9. Target analysis

In [ ]:
if target_column is None:
    print("No target column found.")
else:
    display(raw_df[target_column].value_counts(dropna=False).rename("count").to_frame())

for name in ["target_distribution.png", "class_balance.png"]:
    path = eda_dir / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))

## 10. SMILES-derived properties analysis

In [ ]:
property_df = gallery_df[[smiles_column, "canonical_smiles", "mol"] + ([target_column] if target_column else [])].copy()
property_df["molecular_weight"] = property_df["mol"].apply(Descriptors.MolWt)
property_df["logp"] = property_df["mol"].apply(Descriptors.MolLogP)
property_df["hbd"] = property_df["mol"].apply(Lipinski.NumHDonors)
property_df["hba"] = property_df["mol"].apply(Lipinski.NumHAcceptors)
property_df["qed"] = property_df["mol"].apply(QED.qed)
property_df["lipinski_pass"] = (
    (property_df["molecular_weight"] <= 500)
    & (property_df["logp"] <= 5)
    & (property_df["hbd"] <= 5)
    & (property_df["hba"] <= 10)
)

summary_df = pd.DataFrame(
    {
        "metric": ["valid unique molecules", "mean molecular weight", "mean logP", "mean QED", "Lipinski pass rate"],
        "value": [
            len(property_df),
            round(property_df["molecular_weight"].mean(), 2),
            round(property_df["logp"].mean(), 2),
            round(property_df["qed"].mean(), 3),
            round(property_df["lipinski_pass"].mean() * 100, 1),
        ],
    }
)
display(summary_df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
sns.histplot(property_df["molecular_weight"], bins=30, ax=axes[0], color="#60A5FA")
axes[0].set_title("Molecular weight distribution")
axes[0].set_xlabel("Molecular weight")

sns.histplot(property_df["logp"], bins=30, ax=axes[1], color="#34D399")
axes[1].set_title("LogP distribution")
axes[1].set_xlabel("LogP")

fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
sns.histplot(property_df["qed"], bins=24, ax=axes[0], color="#F59E0B")
axes[0].set_title("QED distribution")
axes[0].set_xlabel("QED")

lipinski_counts = property_df["lipinski_pass"].map({True: "Pass", False: "Fail"}).value_counts()
sns.barplot(x=lipinski_counts.index.tolist(), y=lipinski_counts.values.tolist(), ax=axes[1], palette=["#22C55E", "#EF4444"])
axes[1].set_title("Lipinski pass / fail")
axes[1].set_xlabel("Rule-of-five result")
axes[1].set_ylabel("Molecules")

fig.tight_layout()
plt.show()